In [ ]:
%load_ext autoreload
%autoreload 2

import anndata as ad
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

adata = ad.read_h5ad("./data/larry/larry_processed.h5ad")
adata

In [ ]:
import joblib
from scripts.VectorFieldEmbedder import VectorFieldEmbedder
from scripts.plotting import *

# Load the object back
emb = joblib.load("./data/larry/larry_embedder.pkl")
emb.gene_names = adata.var_names

print("Loaded object:", type(emb))

In [ ]:
import numpy as np

# Embedded coordinates and velocity
X_emb = emb.X_emb              # (N, d)
V_emb = emb.V_emb              # (N, d)

# Pullback metric at embedded points
# metric shape: (N, d, d)
metric = emb.tps.compute_metric(X_emb)

# Metric-induced velocity norm: sqrt(v^T g v)
vnorm = np.sqrt(
    np.einsum("ni,nij,nj->n", V_emb, metric, V_emb)
)
vnorm = np.log(vnorm)
# Robust clipping for color scale
vnorm_clip = np.clip(
    vnorm,
    np.percentile(vnorm, 1),
    np.percentile(vnorm, 95)
)
from scripts.plotting import plot_2d

plot_2d(
    points=X_emb,
    points_color=vnorm_clip,
    cmap="magma",
    title="log(Velocity magnitude) on FlowMap manifold",
    force_continuous=True,
    show_axes=False,
    figsize=(5, 5),
    s=6,
    alpha=0.8
)

In [ ]:
from scripts.VectorFieldGeometry import FixedPointAnalyzer

# ---------------------------------------------------
# Run fixed point analysis
# ---------------------------------------------------
fpa = FixedPointAnalyzer(emb)

fp_info = fpa.identify_fixed_points(
    grid_size=60,
    speed_smooth_sigma=2.3,
    radius_percent=0.1,
)

print(f"Found {len(fp_info)} fixed points")

for i, info in enumerate(fp_info):
    print(f"\nFixed point {i}")
    print("  position:", info["position"])
    print("  type:", info["type"])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.neighbors import NearestNeighbors
from scripts.plotting import compute_velocity_on_grid
from matplotlib.lines import Line2D

# --------------------------------------------------------------------------
# FIXED POINTS (DEFINE ONCE, USE EVERYWHERE)
# --------------------------------------------------------------------------
fp1 = fp_info[0]["position"]   # Fixed point 1 (source)
fp2 = fp_info[1]["position"]   # Fixed point 2 (saddle)

# --------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------
def points_inside_mask(X_emb, seeds, k=8, radius_scale=1.2):
    nn = NearestNeighbors(n_neighbors=k).fit(X_emb)
    r = np.median(nn.kneighbors(X_emb, n_neighbors=k)[0][:, -1]) * radius_scale
    neigh_idx = nn.radius_neighbors(seeds, radius=r, return_distance=False)
    return np.array([len(ix) > 0 for ix in neigh_idx])


def add_manual_arrows(ax, coords, X_emb, V_pred):
    nn = NearestNeighbors(n_neighbors=1).fit(X_emb)
    _, idx = nn.kneighbors(coords)
    idx = idx.ravel()

    ax.quiver(
        X_emb[idx, 0], X_emb[idx, 1],
        V_pred[idx, 0], V_pred[idx, 1],
        angles="xy", scale_units="xy", scale=3,
        width=0.003,
        headwidth=4.5, headlength=4.0, headaxislength=2.3,
        minlength=0.2,
        color="k", alpha=0.9
    )


# --------------------------------------------------------------------------
# 1) Embedding & labels
# --------------------------------------------------------------------------
X_emb = emb.X_emb
labels = np.asarray(adata.obs["state_info"].values)

# --------------------------------------------------------------------------
# 2) Colors
# --------------------------------------------------------------------------
uniq = np.unique(labels)
other = [lab for lab in uniq if lab != "Undifferentiated"]
cmap = plt.get_cmap("tab10", len(other))
colmap = {lab: mcolors.to_hex(cmap(i)) for i, lab in enumerate(other)}
colmap["Undifferentiated"] = "#d3d3d3"
cell_colors = np.array([colmap[lab] for lab in labels])

# --------------------------------------------------------------------------
# 3) Velocity grid
# --------------------------------------------------------------------------
Xg, keep_mass, _ = compute_velocity_on_grid(
    X_emb, grid_size=25, min_mass=0.01
)
keep_inside = points_inside_mask(X_emb, Xg)
Xg = Xg[keep_inside]
Vg = emb.tps_vf.predict(Xg)

# --------------------------------------------------------------------------
# 4) Plot
# --------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 12))

# Background: Undifferentiated
if "Undifferentiated" in uniq:
    m = labels == "Undifferentiated"
    ax.scatter(
        X_emb[m, 0], X_emb[m, 1],
        c="#d3d3d3", s=40, alpha=0.2, linewidths=0
    )

# Foreground cells
m = labels != "Undifferentiated"
ax.scatter(
    X_emb[m, 0], X_emb[m, 1],
    c=cell_colors[m], s=80, alpha=0.4, linewidths=0
)

# Velocity field
ax.quiver(
    Xg[:, 0], Xg[:, 1],
    Vg[:, 0], Vg[:, 1],
    angles="xy", scale_units="xy", scale=3,
    width=0.003,
    headwidth=4.5, headlength=4.0, headaxislength=2.3,
    minlength=0.2,
    color="k", alpha=0.9
)

# --------------------------------------------------------------------------
# 5) Fixed points (explicit, no indices)
# --------------------------------------------------------------------------
dx, dy = -0.02, -0.03

# Fixed point 1
ax.scatter(fp1[0], fp1[1], color="red", s=1000, zorder=4)
ax.text(
    fp1[0] + dx, fp1[1] + dy, "1",
    color="white", fontsize=36, weight="bold",
    ha="center", va="center", zorder=5
)

# Fixed point 2
ax.scatter(fp2[0], fp2[1], color="red", s=1000, zorder=4)
ax.text(
    fp2[0] + dx, fp2[1] + dy, "2",
    color="white", fontsize=36, weight="bold",
    ha="center", va="center", zorder=5
)

# --------------------------------------------------------------------------
# 6) Manual arrows (optional patches)
# --------------------------------------------------------------------------
patch_coords = [
    [12.0, 10.4],
    [14.0, 4.0],
    [13.0, 4.0],
]

V_cells = emb.tps_vf.predict(X_emb)
add_manual_arrows(ax, patch_coords, X_emb, V_cells)

# --------------------------------------------------------------------------
# 7) Formatting
# --------------------------------------------------------------------------
ax.set_aspect("equal")
ax.set_xticks([]); ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
def cells_near_fp(fp, X_2d, epsilon=0.20):
    emb_range = np.ptp(X_2d, axis=0)
    x0, y0 = map(float, fp)
    eps_x = epsilon * emb_range[0]
    eps_y = epsilon * emb_range[1]
    xlim = (x0 - eps_x, x0 + eps_x)
    ylim = (y0 - eps_y, y0 + eps_y)

    mask = (
        (X_2d[:, 0] >= xlim[0]) & (X_2d[:, 0] <= xlim[1]) &
        (X_2d[:, 1] >= ylim[0]) & (X_2d[:, 1] <= ylim[1])
    )
    cell_idx = np.where(mask)[0]
    if len(cell_idx) == 0:
        return None
    return cell_idx, (xlim, ylim)

from scripts.TPS import ThinPlateSpline

emb.X_gene = emb.X
emb.V_gene = emb.V
emb.tps_gene = emb.tps
emb.tps_gene_vf = emb.tps_vf

from scripts.FieldReconstructionEvaluator import FieldReconstructionEvaluator

def eval_pc_concordance_for_region(emb, cell_idx):
    evaluator = FieldReconstructionEvaluator(emb, cell_idx=cell_idx)
    res = evaluator.evaluate_gene_fit()
    return res


cell_idx_fp1, _ = cells_near_fp(fp1, emb.X_emb, epsilon=0.10)
cell_idx_fp2, _ = cells_near_fp(fp2, emb.X_emb, epsilon=0.10)

res_fp1 = eval_pc_concordance_for_region(emb, cell_idx_fp1)
res_fp2 = eval_pc_concordance_for_region(emb, cell_idx_fp2)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# --------------------------------------------------------------------------
# Colors
# --------------------------------------------------------------------------
labels = np.asarray(adata.obs["state_info"].values)
uniq = np.unique(labels)
other = [lab for lab in uniq if lab != "Undifferentiated"]

cmap = plt.get_cmap("tab10", len(other))
colmap = {lab: mcolors.to_hex(cmap(i)) for i, lab in enumerate(other)}
colmap["Undifferentiated"] = "#9e9e9e"

cell_colors = np.array([colmap[lab] for lab in labels])

# --------------------------------------------------------------------------
# Fixed-point semantics (single source of truth)
# --------------------------------------------------------------------------
fp_meta = {
    0: {"label": 1, "kind": "Source", "quiver_scale": 20},
    1: {"label": 2, "kind": "Saddle", "quiver_scale": 10},
}

radius_percent = 0.3

# --------------------------------------------------------------------------
# One figure per fixed point
# --------------------------------------------------------------------------
for fp_idx, meta in fp_meta.items():
    fig, ax = plt.subplots(figsize=(5, 5))

    label = meta["label"]
    kind  = meta["kind"]
    scale = meta["quiver_scale"]

    fp = fp_info[fp_idx]["position"]

    # ----------------------------------------------------------------------
    # Local metric & flattening
    # ----------------------------------------------------------------------
    g_fp = emb.tps.compute_metric(fp[None, :])[0]
    L = np.linalg.cholesky(g_fp)

    dx = emb.X_emb - fp
    dist2 = np.einsum("ni,ij,nj->n", dx, g_fp, dx)

    emb_range = np.ptp(emb.X_emb, axis=0)
    radius = radius_percent * np.mean(emb_range)
    mask = dist2 < radius**2

    Y_loc  = emb.X_emb[mask]
    V_gene = emb.V[mask]
    C_loc  = cell_colors[mask]

    # ----------------------------------------------------------------------
    # Project gene velocity → embedding coordinates
    # ----------------------------------------------------------------------
    V_emb = []
    for y, v in zip(Y_loc, V_gene):
        J = emb.tps.compute_jacobians(y[None, :])[0]
        V_emb.append(np.linalg.pinv(J) @ v)
    V_emb = np.asarray(V_emb)

    # ----------------------------------------------------------------------
    # Flatten to Euclidean normal coordinates
    # ----------------------------------------------------------------------
    X_flat = (Y_loc - fp) @ L.T
    V_flat = V_emb @ L.T

    # ----------------------------------------------------------------------
    # Adaptive square window
    # ----------------------------------------------------------------------
    lim = 0.5 * np.max(np.abs(X_flat))

    # ----------------------------------------------------------------------
    # Plot cells
    # ----------------------------------------------------------------------
    ax.scatter(
        X_flat[:, 0], X_flat[:, 1],
        c=C_loc, s=30, alpha=0.4,
        linewidths=0, zorder=1,
    )

    # ----------------------------------------------------------------------
    # Plot velocity arrows
    # ----------------------------------------------------------------------
    ax.quiver(
        X_flat[:, 0], X_flat[:, 1],
        V_flat[:, 0], V_flat[:, 1],
        color=C_loc,
        angles="xy", scale_units="xy",
        scale=scale,
        width=0.004,
        headwidth=3, headlength=5, headaxislength=4,
        alpha=1.0, zorder=2,
    )

    # ----------------------------------------------------------------------
    # Fixed point at origin
    # ----------------------------------------------------------------------
    ax.scatter(0.0, 0.0, color="red", s=300, zorder=4)
    ax.text(
        -0.01, -0.01, str(label),
        color="white", fontsize=18, weight="bold",
        ha="center", va="center", zorder=5,
    )

    # ----------------------------------------------------------------------
    # Formatting
    # ----------------------------------------------------------------------
    ax.set_aspect("equal")
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)

    ax.set_title(f"Fixed Point {label}: {kind}", fontsize=12)
    ax.set_xlabel("Local linear coord 1")
    ax.set_ylabel("Local linear coord 2")

    plt.tight_layout()
    plt.show()
    break

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------------
n_pcs = 20   # top PCs to analyze

# Fixed-point semantics (same as everywhere else)
fp_meta = {
    0: {"label": 1, "kind": "Source"},
    1: {"label": 2, "kind": "Saddle"},
}

# --------------------------------------------------------------------------
# Extract top PCs (gene space)
# --------------------------------------------------------------------------
# emb.X : (n_cells, n_pcs_total)
# PCs live in gene space; columns are PC directions
PCs = emb.X.T[:, :n_pcs]    # shape (p, n_pcs)

# normalize explicitly (defensive)
PCs /= np.linalg.norm(PCs, axis=0, keepdims=True)

# --------------------------------------------------------------------------
# Compute variance capture for each fixed point
# --------------------------------------------------------------------------
results = {}

for fp_idx, meta in fp_meta.items():
    label = meta["label"]

    fp = fp_info[fp_idx]["position"]

    # Jacobian of embedding map at fixed point: (p, d)
    J = emb.tps.compute_jacobians(fp[None, :])[0]

    # Tangent-space projection operator
    # P = J (J^T J)^(-1) J^T
    P = J @ np.linalg.pinv(J.T @ J) @ J.T

    # Fraction of each PC captured by tangent space
    frac = np.array([
        np.linalg.norm(P @ PCs[:, k])**2
        for k in range(n_pcs)
    ])

    results[f"FP{label}"] = frac

# --------------------------------------------------------------------------
# Assemble DataFrame
# --------------------------------------------------------------------------
df = pd.DataFrame(results, index=np.arange(1, n_pcs + 1))
df.index.name = "PC"

display(df)

# --------------------------------------------------------------------------
# Plot: bar plots (top 20 PCs)
# --------------------------------------------------------------------------
fig, axes = plt.subplots(
    1, len(df.columns),
    figsize=(4 * len(df.columns), 4),
    sharey=True,
)

if len(df.columns) == 1:
    axes = [axes]

for ax, col in zip(axes, df.columns):
    ax.bar(
        df.index,
        df[col].values,
        color="black",
        width=0.8,
    )
    ax.set_title(col)
    ax.set_xlabel("PC")
    ax.set_ylim(0, 1)
    ax.set_xticks(df.index)

axes[0].set_ylabel("Fraction of variance captured")

plt.tight_layout()
plt.show()

In [ ]:
emb.fit_gene_level_splines(dof_gene=50, dof_vf_gene=50)

In [ ]:
genes_to_plot = ["Ltf", "Mmp12", "Tph1", "Ltbp1"]
genes_to_plot = ["Ctss", "Slc6a4", "Vcan", "Gpnmb"]
genes_to_plot = ["Rsad2", "Ddc", "Atp6v0d2", "Slc6a4"]

# --- get predicted expression on manifold ---
pred_expr = emb.tps_gene.predict(emb.X_emb)  # shape (n_cells, n_genes)
gene_names = emb.tps_gene.gene_names if hasattr(emb.tps_gene, "gene_names") else adata.var_names

# collect all predicted values for consistent color scaling
all_expr = []
for gene in genes_to_plot:
    expr = pred_expr[:, np.where(gene_names == gene)[0][0]]
    all_expr.append(expr)
all_expr = np.concatenate(all_expr)
vmin, vmax = np.min(all_expr), np.max(all_expr)

# --- plot ---
fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharex=True, sharey=True)

last_scatter = None
for ax, gene in zip(axes, genes_to_plot):
    expr = pred_expr[:, np.where(gene_names == gene)[0][0]]

    plot_velocity_streamplot(
        emb.X_emb,
        emb.tps_vf,
        scatter_color=expr,
        ax=ax,
        cmap="magma_r",     # light → low, dark → high
        vmin=vmin, vmax=vmax
    )

    ax.set_title(gene, fontsize=44)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel(""); ax.set_ylabel("")
    ax.grid(False); ax.set_frame_on(False)

    last_scatter = ax.collections[-1]

# shared colorbar
# fig.colorbar(last_scatter, ax=axes, orientation='vertical', fraction=0.02, pad=0.02)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors

# --------------------------------------------------------------------------
# Genes to plot (edit freely)
# --------------------------------------------------------------------------
genes_to_plot = ["Ltf", "Mmp12", "Tph1", "Ltbp1"]
# genes_to_plot = ["Ctss", "Slc6a4", "Vcan", "Gpnmb"]
# genes_to_plot = ["Rsad2", "Ddc", "Atp6v0d2", "Slc6a4"]

# --------------------------------------------------------------------------
# Get raw expression
# --------------------------------------------------------------------------
gene_names = adata.var_names.to_numpy()

X_raw = adata.X
if hasattr(X_raw, "toarray"):
    X_raw = X_raw.toarray()

# --------------------------------------------------------------------------
# Shared color scale (across the 4 genes)
# --------------------------------------------------------------------------
all_expr = []
for gene in genes_to_plot:
    idx = np.where(gene_names == gene)[0][0]
    all_expr.append(X_raw[:, idx])

all_expr = np.concatenate(all_expr)
vmin, vmax = np.min(all_expr), np.max(all_expr)

def truncate_colormap(cmap, minval=0.15, maxval=1.0, n=256):
    return colors.LinearSegmentedColormap.from_list(
        "trunc_" + cmap.name,
        cmap(np.linspace(minval, maxval, n))
    )

cmap_dark = truncate_colormap(cm.magma_r, minval=0.02)

# --------------------------------------------------------------------------
# Plot: raw expression only
# --------------------------------------------------------------------------
fig, axes = plt.subplots(
    1, len(genes_to_plot),
    figsize=(5 * len(genes_to_plot), 5),
    sharex=True, sharey=True
)

if len(genes_to_plot) == 1:
    axes = [axes]

last_scatter = None

for ax, gene in zip(axes, genes_to_plot):
    idx = np.where(gene_names == gene)[0][0]

    sc = ax.scatter(
        emb.X_emb[:, 0],
        emb.X_emb[:, 1],
        c=X_raw[:, idx],
        cmap=cmap_dark,
        s=12,
        alpha=0.5,
        vmin=vmin,
        vmax=vmax,
        linewidths=0
    )

    ax.set_title(gene, fontsize=36)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_frame_on(False)

    last_scatter = sc

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# --------------------------------------------------------------------------
# 1️⃣ Cell-type coloring setup
# --------------------------------------------------------------------------
labels = np.asarray(adata.obs["state_info"].values)

uniq = np.unique(labels)
other = [lab for lab in uniq if lab != "Undifferentiated"]
cmap = plt.get_cmap("tab10", len(other))
colmap = {lab: mcolors.to_hex(cmap(i)) for i, lab in enumerate(other)}
colmap["Undifferentiated"] = "#d3d3d3"

cell_colors = np.array([colmap[lab] for lab in labels])

# --------------------------------------------------------------------------
# 2️⃣ Phase portraits per gene (2 × 6 grid)
# --------------------------------------------------------------------------
genes_to_plot = [
    "Ltf", "Mmp12", "Tph1", "Ltbp1",
    "Ctss", "Slc6a4", "Vcan", "Gpnmb",
    "Rsad2", "Ddc", "Atp6v0d2", "Slc6a4",
]

U = adata.layers["Mu"]
S = adata.layers["Ms"]

fig, axes = plt.subplots(2, 6, figsize=(24, 8))
axes = axes.flatten()

for ax, gene in zip(axes, genes_to_plot):
    idx = np.where(adata.var_names == gene)[0][0]

    # convert sparse → dense safely
    u = U[:, idx]
    s = S[:, idx]
    u = u.toarray().ravel() if hasattr(u, "toarray") else np.ravel(u)
    s = s.toarray().ravel() if hasattr(s, "toarray") else np.ravel(s)

    # scatter: colored by cell type
    ax.scatter(
        u, s,
        s=10,
        alpha=0.5,
        c=cell_colors,
        edgecolor="none"
    )

    ax.set_title(gene, fontsize=18)
    ax.set_xlabel("Unspliced")
    ax.set_ylabel("Spliced")

    # steady-state fit line
    mask = (u > 0) & (s > 0)
    if np.sum(mask) > 10:
        coef = np.polyfit(u[mask], s[mask], 1)
        x_fit = np.linspace(u[mask].min(), u[mask].max(), 100)
        ax.plot(
            x_fit,
            np.polyval(coef, x_fit),
            color="darkorange",
            lw=2
        )

# turn off unused axes if fewer than 12 genes
for ax in axes[len(genes_to_plot):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

# --- 1) Use first 3 PCs ---
raw_expr = np.asarray(emb.X[:, :3])
pred_expr = np.asarray(emb.tps.predict(emb.X_emb)[:, :3])

# --- 2) Define cell state colors ---
uniq = np.unique(labels)
other = [lab for lab in uniq if lab != "Undifferentiated"]
cmap = plt.get_cmap("tab10", len(other))
colmap = {lab: mcolors.to_hex(cmap(i)) for i, lab in enumerate(other)}
colmap["Undifferentiated"] = "#d3d3d3"
cell_colors = np.array([colmap[lab] for lab in labels])

# --- 3) Helper: polished 3D scatter with ordered overlay ---
def scatter3d(data, labels, colors, title):
    fig = plt.figure(figsize=(6, 5), facecolor="white")
    ax = fig.add_subplot(111, projection="3d")

    x, y, z = data.T

    # Plot Undifferentiated first (underneath)
    mask_undiff = (labels == "Undifferentiated")
    mask_other = ~mask_undiff

    # Undifferentiated (background layer)
    ax.scatter(x[mask_undiff], y[mask_undiff], z[mask_undiff],
               c=colors[mask_undiff], s=30, alpha=0.35, edgecolors='none')

    # Other states (foreground)
    ax.scatter(x[mask_other], y[mask_other], z[mask_other],
               c=colors[mask_other], s=8, alpha=0.9, edgecolors='none')

    # Aesthetics
    ax.set_title(title, fontsize=12, pad=10)
    ax.view_init(elev=25, azim=-130)
    
#     ax.set_xlabel("PC1", labelpad=8)
#     ax.set_ylabel("PC2", labelpad=8)
#     ax.set_zlabel("PC3", labelpad=8)

    # --- Zoom out slightly ---
    xlim, ylim, zlim = ax.get_xlim(), ax.get_ylim(), ax.get_zlim()
    zoom = 0.7  # zoom-out factor
    ax.set_xlim(np.array(xlim) * zoom)
    ax.set_ylim(np.array(ylim) * zoom)
    ax.set_zlim(np.array(zlim) * zoom)

    ax.set_facecolor("white")
    ax.grid(False)
    for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
        axis.set_ticklabels([])
        axis.line.set_color("black")

    plt.tight_layout()
    plt.show()

# --- 4) Plot separately with overlay order ---
scatter3d(raw_expr, labels, cell_colors, "")
scatter3d(pred_expr, labels, cell_colors, "")

In [ ]:
# --- 1) Use first 4-6 PCs ---
# PCs 4–6
raw_expr = np.asarray(emb.X[:, 3:6])
pred_expr = np.asarray(emb.tps.predict(emb.X_emb)[:, 3:6])

# --- 2) Define cell state colors ---
uniq = np.unique(labels)
other = [lab for lab in uniq if lab != "Undifferentiated"]
cmap = plt.get_cmap("tab10", len(other))
colmap = {lab: mcolors.to_hex(cmap(i)) for i, lab in enumerate(other)}
colmap["Undifferentiated"] = "#d3d3d3"
cell_colors = np.array([colmap[lab] for lab in labels])

# --- 3) Helper: polished 3D scatter with ordered overlay ---
def scatter3d(data, labels, colors, title):
    fig = plt.figure(figsize=(6, 8), facecolor="white")
    ax = fig.add_subplot(111, projection="3d")

    x, y, z = data.T

    # Plot Undifferentiated first (underneath)
    mask_undiff = (labels == "Undifferentiated")
    mask_other = ~mask_undiff

    # Undifferentiated (background layer)
    ax.scatter(x[mask_undiff], y[mask_undiff], z[mask_undiff],
               c=colors[mask_undiff], s=30, alpha=0.35, edgecolors='none')

    # Other states (foreground)
    ax.scatter(x[mask_other], y[mask_other], z[mask_other],
               c=colors[mask_other], s=8, alpha=0.9, edgecolors='none')

    # Aesthetics
    ax.set_title(title, fontsize=12, pad=10)
    ax.view_init(elev=25, azim=-130)
    
    ax.set_xlabel("PC4", labelpad=8)
    ax.set_ylabel("PC5", labelpad=8)
    ax.set_zlabel("PC6", labelpad=8)

    # --- Zoom out slightly ---
    xlim, ylim, zlim = ax.get_xlim(), ax.get_ylim(), ax.get_zlim()
    zoom = 0.7  # zoom-out factor
    ax.set_xlim(np.array(xlim) * zoom)
    ax.set_ylim(np.array(ylim) * zoom)
    ax.set_zlim(np.array(zlim) * zoom)

    ax.set_facecolor("white")
    ax.grid(False)
    for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
        axis.set_ticklabels([])
        axis.line.set_color("black")

    plt.tight_layout()
    plt.show()

# --- 4) Plot separately with overlay order ---
scatter3d(raw_expr, labels, cell_colors, "")
scatter3d(pred_expr, labels, cell_colors, "")

In [ ]:
# --- 1) Use first 7-9 PCs ---
# PCs 7–9 (for supplementary)
raw_expr = np.asarray(emb.X[:, 6:9])
pred_expr = np.asarray(emb.tps.predict(emb.X_emb)[:, 6:9])

# --- 2) Define cell state colors ---
uniq = np.unique(labels)
other = [lab for lab in uniq if lab != "Undifferentiated"]
cmap = plt.get_cmap("tab10", len(other))
colmap = {lab: mcolors.to_hex(cmap(i)) for i, lab in enumerate(other)}
colmap["Undifferentiated"] = "#d3d3d3"
cell_colors = np.array([colmap[lab] for lab in labels])

# --- 3) Helper: polished 3D scatter with ordered overlay ---
def scatter3d(data, labels, colors, title):
    fig = plt.figure(figsize=(6, 5), facecolor="white")
    ax = fig.add_subplot(111, projection="3d")

    x, y, z = data.T

    # Plot Undifferentiated first (underneath)
    mask_undiff = (labels == "Undifferentiated")
    mask_other = ~mask_undiff

    # Undifferentiated (background layer)
    ax.scatter(x[mask_undiff], y[mask_undiff], z[mask_undiff],
               c=colors[mask_undiff], s=30, alpha=0.35, edgecolors='none')

    # Other states (foreground)
    ax.scatter(x[mask_other], y[mask_other], z[mask_other],
               c=colors[mask_other], s=8, alpha=0.9, edgecolors='none')

    # Aesthetics
    ax.set_title(title, fontsize=12, pad=10)
    ax.view_init(elev=25, azim=-130)
    
    ax.set_xlabel("PC7", labelpad=8)
    ax.set_ylabel("PC8", labelpad=8)
    ax.set_zlabel("PC9", labelpad=8)

    # --- Zoom out slightly ---
    xlim, ylim, zlim = ax.get_xlim(), ax.get_ylim(), ax.get_zlim()
    zoom = 0.7  # zoom-out factor
    ax.set_xlim(np.array(xlim) * zoom)
    ax.set_ylim(np.array(ylim) * zoom)
    ax.set_zlim(np.array(zlim) * zoom)

    ax.set_facecolor("white")
    ax.grid(False)
    for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
        axis.set_ticklabels([])
        axis.line.set_color("black")

    plt.tight_layout()
    plt.show()

# --- 4) Plot separately with overlay order ---
scatter3d(raw_expr, labels, cell_colors, "")
scatter3d(pred_expr, labels, cell_colors, "")